# Bảng feature của mô hình triển khai

## 1. Mục đích và phạm vi

Mô hình ước lượng **nhiệt độ trung bình kỳ vọng theo tháng** cho một thành phố. Người dùng chỉ chọn thành phố, tháng và năm; tên thành phố chỉ dùng để tra cứu thuộc tính phù hợp, không được đưa trực tiếp vào mô hình.

Tập đầu vào chính thức gồm **15 feature**: 14 feature số và 1 feature phân loại. Tất cả có thể xác định trước khi dự báo, không cần nhiệt độ tháng trước.

- `static`: xác định từ tháng/năm mục tiêu hoặc thuộc tính cố định của thành phố.
- `fit_stat`: thống kê rút ra từ dữ liệu lịch sử, không cần nhiệt độ của tháng đang dự báo.

## 2. Feature được giữ lại để huấn luyện và triển khai

| # | Feature | Công thức / nguồn | Nhóm | Khả dụng | Vai trò |
|---:|---|---|---|---|---|
| 1 | `month_sin` | `sin(2π × month / 12)` | Mùa vụ | `static` | Mã hóa chu kỳ tháng. |
| 2 | `month_cos` | `cos(2π × month / 12)` | Mùa vụ | `static` | Đi cùng sin để giữ pha mùa đầy đủ. |
| 3 | `climatic_month_sin` | `sin(2π × climatic_month / 12)` | Mùa vụ/bán cầu | `static` | Hiệu chỉnh pha mùa giữa hai bán cầu. |
| 4 | `climatic_month_cos` | `cos(2π × climatic_month / 12)` | Mùa vụ/bán cầu | `static` | Đi cùng sin để giữ pha mùa đầy đủ. |
| 5 | `years_since_start` | `year − 1863` | Xu hướng | `static` | Biểu diễn xu hướng thời gian dài hạn. |
| 6 | `latitude` | Vĩ độ có dấu | Địa lý | `static` | Phân biệt vị trí Bắc/Nam. |
| 7 | `abs_latitude` | `abs(latitude)` | Địa lý | `static` | Khoảng cách tới xích đạo. |
| 8 | `hemisphere_north` | `latitude ≥ 0` | Địa lý | `static` | Cờ Bắc/Nam bán cầu. |
| 9 | `longitude` | Kinh độ thành phố | Địa lý | `static` | Bối cảnh vị trí đông–tây. |
| 10 | `abslat_x_month_sin` | `abs_latitude × climatic_month_sin` | Tương tác | `static` | Biên độ mùa theo vĩ độ. |
| 11 | `abslat_x_month_cos` | `abs_latitude × climatic_month_cos` | Tương tác | `static` | Thành phần pha thứ hai của tương tác. |
| 12 | `loc_month_climatology` | Trung bình lịch sử theo thành phố–tháng | Khí hậu TP | `fit_stat` | Nhiệt độ khí hậu nền; tín hiệu mạnh nhất. |
| 13 | `loc_mean_temperature` | Trung bình lịch sử của thành phố | Khí hậu TP | `fit_stat` | Nền nhiệt dài hạn. |
| 14 | `loc_temperature_std` | Độ lệch chuẩn lịch sử của thành phố | Khí hậu TP | `fit_stat` | Mức biến động nhiệt độ đặc trưng. |
| 15 | `country_name` | Quốc gia thành phố | Phân loại | `static` | One-hot encoding trong Pipeline. |

**Tháng khí hậu:** Bắc bán cầu giữ nguyên tháng; Nam bán cầu dịch 6 tháng: `((month + 5) mod 12) + 1`.

## 3. Kiểm soát leakage

- Khi đánh giá validation/test, thống kê khí hậu theo thành phố chỉ được fit từ dữ liệu trước năm 1984; `loc_month_climatology` dùng leave-one-out ở giai đoạn fit.
- Khi triển khai, thống kê được tính sẵn từ lịch sử có đến 09/2013; chúng không dùng nhiệt độ của tháng tương lai cần dự báo.
- `country_name` được one-hot bên trong Pipeline và encoder chỉ fit trên train.
- Không dùng `city_name` làm feature để tránh mô hình ghi nhớ hơn 3.000 thành phố; tên thành phố chỉ là khóa tra cứu tọa độ và climatology.

## 4. Feature bị loại khỏi mô hình triển khai

| Nhóm / feature | Lý do loại |
|---|---|
| `temp_lag_1`, `temp_lag_12` | Cần nhiệt độ quan trắc các tháng trước; dữ liệu hiện có dừng ở 09/2013. |
| Rolling/anomaly (`temp_roll_*`, `temp_anomaly_lag_12`) | Cần chuỗi quan trắc gần nhất; dự báo đệ quy sẽ tích lũy sai số. |
| `land_temperature_lag_1`, `land_anomaly_lag_1` | Cần số liệu toàn cầu cập nhật gần forecast origin. |
| `city_uncertainty_lag_1` | Không có sẵn cho tháng tương lai. |
| Nhiệt độ quốc gia/thành phố lớn cùng tháng | Nguy cơ leakage với target cùng tháng. |
| `is_major_city`, `abslat_x_years`, `loc_climatology_imputed` | Tín hiệu quá yếu, trùng lặp mạnh hoặc gần hằng số. |

## 5. Cách diễn giải mô hình

Mô hình học **quy luật khí hậu lịch sử của từng thành phố**: nền nhiệt theo tháng, mùa vụ, vị trí địa lý, biến động lịch sử và xu hướng thời gian. Vì dữ liệu quan trắc kết thúc tại 09/2013, kết quả sau năm 2013 là **ước lượng nhiệt độ trung bình khí hậu kỳ vọng**, không phải nhiệt độ quan trắc đã được xác minh.

Nguồn đối chiếu: `notebooks_v1/05_feature_engineering.ipynb`, `data/processed/feature_metadata.json`, `models/model_metadata.json`.